# Chapter 2: Simple Linear Regression
### Heiss & Brunner — *Using Python for Introductory Econometrics* (2024)

---

## How this notebook works
Each step is in its **own cell** — read the explanation, then run the cell.

For every calculation we will:
1. Do it **manually** (step by step, from scratch)
2. Repeat it with **`statsmodels`**
3. **Compare** the two results to confirm they match

## Topics
1. The simple regression model — what it is and why we use it
2. Computing OLS estimates manually (beta0 and beta1)
3. Fitted values and residuals
4. R-squared
5. Standard errors and t-tests — manually and with statsmodels
6. Confidence intervals
7. Log transformations
8. Visualising the regression line and residuals
9. Monte Carlo: verifying OLS is unbiased

## Cell 1 — Setup: imports and working directory

In [ ]:
import os
import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
import scipy.stats as stats
import statsmodels.formula.api as smf   # formula-based OLS
import statsmodels.api as sm            # array-based OLS
import wooldridge as woo

In [ ]:
# Fix working directory to this chapter's folder
chapter_dir = os.path.dirname(os.path.abspath('__file__'))
os.chdir(chapter_dir)

# Create output folder
output_dir = os.path.join(chapter_dir, 'output')
os.makedirs(output_dir, exist_ok=True)

print('Working directory:', os.getcwd())
print('Output directory :', output_dir)

---
## Cell 2 — Load the data and look at it

We use the **`wage1`** dataset from Wooldridge (526 US workers, 1976 CPS).

**Before any regression, always look at your data first.**

In [ ]:
# Load the dataset
df = woo.dataWoo('wage1')

print('Number of observations:', len(df))
print('Number of variables   :', df.shape[1])

In [ ]:
# Look at the first 10 rows — what does each row represent?
# Each row = one worker
# wage = hourly wage in dollars
# educ = years of education
print('First 10 rows of the raw dataset:')
print(df[['wage', 'educ', 'exper', 'tenure', 'female']].head(10).to_string())

In [ ]:
# Summary statistics for wage and educ — the two variables we will use
print('Summary statistics:')
print(df[['wage', 'educ']].describe().round(3).to_string())

In [ ]:
# Quick scatter plot: does wage increase with education?
plt.figure(figsize=(8, 5))
plt.scatter(df['educ'], df['wage'], alpha=0.35, color='steelblue', s=20)
plt.xlabel('Years of Education', fontsize=12)
plt.ylabel('Hourly Wage ($/hour)', fontsize=12)
plt.title('Wage vs Education (raw data)', fontsize=13, fontweight='bold')
plt.savefig(os.path.join(output_dir, 'fig_ch02_01_scatter_raw.png'), dpi=150, bbox_inches='tight')
plt.show()

print('Observation: positive relationship between education and wage,')
print('but a lot of scatter (other factors also matter).')

---
## Section 1 — The Simple Regression Model

### WHY simple regression?
The scatter plot shows a positive relationship, but how large is it exactly?
Simple regression fits the best straight line through the cloud of points.

### The model
$$y_i = \beta_0 + \beta_1 x_i + u_i, \quad i = 1, \ldots, n$$

- $y_i$ = **wage** (what we want to explain)
- $x_i$ = **educ** (what we use to explain it)
- $\beta_0$ = **intercept** — predicted wage when educ = 0
- $\beta_1$ = **slope** — change in wage per extra year of education
- $u_i$ = **error term** — everything else affecting wage (ability, luck, etc.)

### The OLS method
We want to find $\hat{\beta}_0$ and $\hat{\beta}_1$ that minimise:
$$SSR = \sum_{i=1}^n \hat{u}_i^2 = \sum_{i=1}^n (y_i - \hat{\beta}_0 - \hat{\beta}_1 x_i)^2$$

Taking the first-order conditions gives the OLS formulas below.

---
## Section 2 — Computing OLS Estimates

### The formulas
$$\hat{\beta}_1 = \frac{\sum_{i=1}^n (x_i - \bar{x})(y_i - \bar{y})}{\sum_{i=1}^n (x_i - \bar{x})^2}$$

$$\hat{\beta}_0 = \bar{y} - \hat{\beta}_1 \bar{x}$$

We will compute this **step by step**.

### Step 2.1 — Extract the variables we need

In [ ]:
# Pull out y and x as NumPy arrays
y = df['wage'].values   # dependent variable (what we explain)
x = df['educ'].values   # independent variable (the regressor)
n = len(y)

print('y (wage) — first 5 values:', y[:5])
print('x (educ) — first 5 values:', x[:5])
print('n (observations)          :', n)

### Step 2.2 — Compute sample means

In [ ]:
y_bar = y.mean()   # mean of wage
x_bar = x.mean()   # mean of education

print('Mean of wage (y_bar):', round(y_bar, 6))
print('Mean of educ (x_bar):', round(x_bar, 6))

### Step 2.3 — Compute deviations from the mean

In [ ]:
# Deviation of each x_i from the mean: (x_i - x_bar)
x_dev = x - x_bar

# Deviation of each y_i from the mean: (y_i - y_bar)
y_dev = y - y_bar

# Show for first 5 workers
print('First 5 workers: x, x_dev, y, y_dev')
for i in range(5):
    print(f'  Worker {i+1}: educ={x[i]:.0f}, x_dev={x_dev[i]:+.4f}, '
          f'wage={y[i]:.2f}, y_dev={y_dev[i]:+.4f}')

### Step 2.4 — Compute the numerator of beta1_hat

In [ ]:
# Numerator = sum of (x_i - x_bar)(y_i - y_bar)
# This is the sample covariance (times n-1)
numerator = np.sum(x_dev * y_dev)

print('Numerator = sum( (x_i - x_bar)(y_i - y_bar) )')
print('         =', round(numerator, 6))

# Also equals n * sample_cov(x, y)
print('Cross-check: (n-1) * cov(x,y) =', round((n-1) * np.cov(x, y)[0,1], 6))

### Step 2.5 — Compute the denominator of beta1_hat

In [ ]:
# Denominator = sum of (x_i - x_bar)^2
# This is proportional to the sample variance of x
denominator = np.sum(x_dev ** 2)

print('Denominator = sum( (x_i - x_bar)^2 )')
print('           =', round(denominator, 6))

# Also equals (n-1) * var(x)
print('Cross-check: (n-1) * var(x) =', round((n-1) * x.var(ddof=1), 6))

### Step 2.6 — Compute beta1_hat (the slope)

In [ ]:
beta1_hat = numerator / denominator

print('beta1_hat = numerator / denominator')
print(f'         = {numerator:.6f} / {denominator:.6f}')
print(f'         = {beta1_hat:.6f}')

### Step 2.7 — Compute beta0_hat (the intercept)

In [ ]:
beta0_hat = y_bar - beta1_hat * x_bar

print('beta0_hat = y_bar - beta1_hat * x_bar')
print(f'         = {y_bar:.6f} - {beta1_hat:.6f} * {x_bar:.6f}')
print(f'         = {beta0_hat:.6f}')

### Step 2.8 — Now do the same with `statsmodels` and compare

We use `smf.ols('wage ~ educ', data=df).fit()` which means:
- `wage` is the dependent variable (y)
- `educ` is the regressor (x)
- `~` separates y from x
- `.fit()` runs the OLS estimation

In [ ]:
# Run OLS with statsmodels formula API
results = smf.ols('wage ~ educ', data=df).fit()

In [ ]:
# Extract the coefficients from statsmodels
beta0_sm = results.params['Intercept']
beta1_sm = results.params['educ']

print('beta0 (Intercept):')
print(f'  Manual     : {beta0_hat:.6f}')
print(f'  statsmodels: {beta0_sm:.6f}')
print(f'  Match?     : {np.isclose(beta0_hat, beta0_sm)}')

print()
print('beta1 (educ slope):')
print(f'  Manual     : {beta1_hat:.6f}')
print(f'  statsmodels: {beta1_sm:.6f}')
print(f'  Match?     : {np.isclose(beta1_hat, beta1_sm)}')

### Interpretation of the coefficients

> Run the cell below to see what the numbers mean in plain English.

In [ ]:
print('INTERPRETATION:')
print(f'  Estimated equation: wage_hat = {beta0_hat:.4f} + {beta1_hat:.4f} * educ')
print()
print(f'  beta0_hat = {beta0_hat:.4f}')
print(f'    -> Predicted wage for a worker with 0 years of education: ${beta0_hat:.2f}/hour')
print(f'    -> (No one in the sample has 0 education, so this is an extrapolation.)')
print()
print(f'  beta1_hat = {beta1_hat:.4f}')
print(f'    -> Each additional year of education is associated with')
print(f'       ${beta1_hat:.4f} higher hourly wage on average.')
print(f'    -> A 4-year university degree (4 extra years) -> +${beta1_hat*4:.2f}/hour on average.')

---
## Section 3 — Fitted Values and Residuals

**Fitted value** $\hat{y}_i = \hat{\beta}_0 + \hat{\beta}_1 x_i$ — model's prediction for worker $i$

**Residual** $\hat{u}_i = y_i - \hat{y}_i$ — how wrong the model is for worker $i$

OLS minimises $\sum \hat{u}_i^2$, so residuals are as small as possible.

### Step 3.1 — Compute fitted values manually

In [ ]:
# Fitted value for every worker: y_hat_i = beta0_hat + beta1_hat * educ_i
y_hat_manual = beta0_hat + beta1_hat * x

print('First 5 fitted values (manual):')
print(y_hat_manual[:5].round(4))

### Step 3.2 — Get fitted values from statsmodels

In [ ]:
y_hat_sm = results.fittedvalues.values

print('First 5 fitted values (statsmodels):')
print(y_hat_sm[:5].round(4))

print('\nAll match?', np.allclose(y_hat_manual, y_hat_sm))

### Step 3.3 — Compute residuals manually

In [ ]:
resid_manual = y - y_hat_manual

print('First 5 residuals (manual):')
print(resid_manual[:5].round(4))

### Step 3.4 — Get residuals from statsmodels

In [ ]:
resid_sm = results.resid.values

print('First 5 residuals (statsmodels):')
print(resid_sm[:5].round(4))

print('\nAll match?', np.allclose(resid_manual, resid_sm))

### Step 3.5 — Show actual, fitted, and residual side by side

In [ ]:
comparison = pd.DataFrame({
    'educ (x)'         : x[:10],
    'wage actual (y)'  : y[:10].round(2),
    'wage fitted (y_hat)' : y_hat_manual[:10].round(4),
    'residual (u_hat)' : resid_manual[:10].round(4)
})

print('First 10 workers — actual vs fitted vs residual:')
print(comparison.to_string(index=False))

### Step 3.6 — Verify the three algebraic properties of OLS residuals

In [ ]:
# Property 1: residuals sum to zero
print('Property 1: sum of residuals = 0?')
print(f'  sum(u_hat) = {resid_manual.sum():.10f}  (approx 0)')

In [ ]:
# Property 2: residuals are orthogonal to x
print('Property 2: sum(x_i * u_hat_i) = 0?')
print(f'  sum(x * u_hat) = {(x * resid_manual).sum():.10f}  (approx 0)')

In [ ]:
# Property 3: regression line passes through (x_bar, y_bar)
print('Property 3: the line passes through (x_bar, y_bar)?')
pred_at_mean = beta0_hat + beta1_hat * x_bar
print(f'  beta0 + beta1 * x_bar = {pred_at_mean:.6f}')
print(f'  y_bar                 = {y_bar:.6f}')
print(f'  Equal?                : {np.isclose(pred_at_mean, y_bar)}')

---
## Section 4 — R-squared (Goodness of Fit)

### WHY R-squared?
R² measures how much of the variation in wages is explained by education.

### The decomposition
$$\underbrace{\sum_i (y_i - \bar{y})^2}_{SST} = \underbrace{\sum_i (\hat{y}_i - \bar{y})^2}_{SSE} + \underbrace{\sum_i \hat{u}_i^2}_{SSR}$$

$$R^2 = \frac{SSE}{SST} = 1 - \frac{SSR}{SST}$$

### Step 4.1 — Compute SST (total variation in y)

In [ ]:
SST = np.sum((y - y_bar)**2)

print('SST = sum( (y_i - y_bar)^2 )')
print(f'    = {SST:.4f}')
print(f'This is (n-1) * var(wage) = {(n-1)*y.var(ddof=1):.4f}  <- same')

### Step 4.2 — Compute SSR (residual variation — what model got WRONG)

In [ ]:
SSR = np.sum(resid_manual**2)

print('SSR = sum( u_hat_i^2 )')
print(f'    = {SSR:.4f}')

### Step 4.3 — Compute SSE (explained variation — what model got RIGHT)

In [ ]:
SSE = SST - SSR

print('SSE = SST - SSR')
print(f'    = {SST:.4f} - {SSR:.4f}')
print(f'    = {SSE:.4f}')

# Cross-check
SSE_check = np.sum((y_hat_manual - y_bar)**2)
print(f'Cross-check: sum( (y_hat - y_bar)^2 ) = {SSE_check:.4f}  <- same')

### Step 4.4 — Compute R-squared manually

In [ ]:
R2_manual = SSE / SST

print('R^2 = SSE / SST')
print(f'    = {SSE:.4f} / {SST:.4f}')
print(f'    = {R2_manual:.6f}')

### Step 4.5 — Get R-squared from statsmodels

In [ ]:
R2_sm = results.rsquared

print('R^2 from statsmodels:', round(R2_sm, 6))
print('R^2 manual          :', round(R2_manual, 6))
print('Match?              :', np.isclose(R2_manual, R2_sm))

### Step 4.6 — Interpret R-squared

In [ ]:
print('INTERPRETATION:')
print(f'  R^2 = {R2_manual:.4f}')
print(f'  -> Education explains {R2_manual*100:.1f}% of the variation in wages.')
print(f'  -> The remaining {(1-R2_manual)*100:.1f}% is due to other factors')
print(f'     (experience, ability, gender, occupation, etc.)')
print()
print('NOTE: In simple regression, R^2 = corr(x, y)^2')
r_xy = np.corrcoef(x, y)[0, 1]
print(f'  corr(educ, wage)^2 = {r_xy:.4f}^2 = {r_xy**2:.4f}  <- same as R^2')

---
## Section 5 — Standard Errors and t-Tests

### WHY do we need standard errors?
We estimated $\hat{\beta}_1 = 0.54$. But this is based on one sample.
A different sample would give a different number.
The standard error tells us: how precise is this estimate?

### The formulas
$$\hat{\sigma}^2 = \frac{SSR}{n - 2} \quad \text{(error variance)}$$

$$\text{se}(\hat{\beta}_1) = \sqrt{\frac{\hat{\sigma}^2}{\sum_i (x_i - \bar{x})^2}}$$

$$t_{\hat{\beta}_1} = \frac{\hat{\beta}_1}{\text{se}(\hat{\beta}_1)} \sim t_{n-2} \text{ under H}_0: \beta_1 = 0$$

### Step 5.1 — Estimate the error variance (sigma-squared)

In [ ]:
# sigma^2_hat = SSR / (n - 2)
# We divide by n-2 because we estimated 2 parameters: beta0 and beta1
sigma2_hat = SSR / (n - 2)

print('sigma2_hat = SSR / (n - 2)')
print(f'           = {SSR:.4f} / {n-2}')
print(f'           = {sigma2_hat:.4f}')

sigma_hat = np.sqrt(sigma2_hat)
print(f'sigma_hat  = sqrt({sigma2_hat:.4f}) = {sigma_hat:.4f}')
print(f'  This is the estimated standard deviation of the error term.')

### Step 5.2 — Compute the standard error of beta1_hat

In [ ]:
# SSx = sum of squared deviations of x = denominator we computed earlier
SSx = denominator   # = sum( (x_i - x_bar)^2 )

se_beta1 = np.sqrt(sigma2_hat / SSx)

print('se(beta1_hat) = sqrt( sigma2_hat / SSx )')
print(f'             = sqrt( {sigma2_hat:.4f} / {SSx:.4f} )')
print(f'             = {se_beta1:.6f}')

### Step 5.3 — Compute the standard error of beta0_hat

In [ ]:
# se(beta0) needs the formula:
# se(beta0_hat) = sigma_hat * sqrt( sum(x_i^2) / (n * SSx) )
se_beta0 = sigma_hat * np.sqrt(np.sum(x**2) / (n * SSx))

print('se(beta0_hat) =', round(se_beta0, 6))

### Step 5.4 — Compute the t-statistics

In [ ]:
# t-stat for beta1: H0: beta1 = 0
t_stat_b1 = beta1_hat / se_beta1

# t-stat for beta0: H0: beta0 = 0
t_stat_b0 = beta0_hat / se_beta0

print('t-statistic for beta1 (educ):')
print(f'  t = beta1_hat / se(beta1_hat) = {beta1_hat:.6f} / {se_beta1:.6f} = {t_stat_b1:.4f}')

print('t-statistic for beta0 (Intercept):')
print(f'  t = beta0_hat / se(beta0_hat) = {beta0_hat:.6f} / {se_beta0:.6f} = {t_stat_b0:.4f}')

### Step 5.5 — Compute p-values

In [ ]:
# p-value = P(|T| > |t_stat|) under H0, with t_{n-2} distribution
# df = n - 2 (two parameters estimated)
df_resid = n - 2

p_val_b1 = 2 * stats.t.sf(abs(t_stat_b1), df=df_resid)
p_val_b0 = 2 * stats.t.sf(abs(t_stat_b0), df=df_resid)

print(f'Degrees of freedom = n - 2 = {df_resid}')
print()
print(f'p-value for beta1: 2 * P(T > |{t_stat_b1:.4f}|) = {p_val_b1:.6f}')
print(f'p-value for beta0: 2 * P(T > |{t_stat_b0:.4f}|) = {p_val_b0:.6f}')

### Step 5.6 — Compare manual results with statsmodels

In [ ]:
# Get the full output from statsmodels
print('Full statsmodels output:')
print(results.summary())

In [ ]:
# Now compare coefficient by coefficient
print('COMPARISON: Manual vs statsmodels')
print('=' * 55)

print(f'{"":<20} {"Manual":>12} {"statsmodels":>12} {"Match?":>8}')
print('-' * 55)

rows = [
    ('beta0_hat',    beta0_hat,    results.params['Intercept']),
    ('beta1_hat',    beta1_hat,    results.params['educ']),
    ('se(beta0)',    se_beta0,     results.bse['Intercept']),
    ('se(beta1)',    se_beta1,     results.bse['educ']),
    ('t-stat beta0', t_stat_b0,   results.tvalues['Intercept']),
    ('t-stat beta1', t_stat_b1,   results.tvalues['educ']),
    ('p-val beta0',  p_val_b0,    results.pvalues['Intercept']),
    ('p-val beta1',  p_val_b1,    results.pvalues['educ']),
    ('R-squared',    R2_manual,   results.rsquared),
    ('SSR',          SSR,         results.ssr),
]
for label, manual, sm_val in rows:
    match = np.isclose(manual, sm_val)
    print(f'{label:<20} {manual:>12.6f} {sm_val:>12.6f} {str(match):>8}')

---
## Section 6 — Confidence Intervals for Coefficients

### Formula
$$\hat{\beta}_j \pm t_{\alpha/2,\; n-2} \cdot \text{se}(\hat{\beta}_j)$$

For 95% confidence: $\alpha = 0.05$, so we use $t_{0.025, n-2}$.

### Step 6.1 — Find the critical value

In [ ]:
alpha   = 0.05
t_crit  = stats.t.ppf(1 - alpha/2, df=n-2)   # 97.5th percentile

print(f'Critical value: t_{{0.025, {n-2}}} = {t_crit:.4f}')
print(f'(95% of the t-distribution falls between -{t_crit:.4f} and +{t_crit:.4f})')

### Step 6.2 — Compute 95% CI for beta1_hat manually

In [ ]:
CI_low_b1  = beta1_hat - t_crit * se_beta1
CI_high_b1 = beta1_hat + t_crit * se_beta1

print('95% CI for beta1_hat (educ):')
print(f'  [{beta1_hat:.6f} - {t_crit:.4f}*{se_beta1:.6f},  {beta1_hat:.6f} + {t_crit:.4f}*{se_beta1:.6f}]')
print(f'  = [{CI_low_b1:.6f}, {CI_high_b1:.6f}]')

### Step 6.3 — Get CI from statsmodels and compare

In [ ]:
ci_sm = results.conf_int(alpha=0.05)

print('Confidence intervals from statsmodels:')
print(ci_sm.round(6))

print()
print('CI for beta1 — comparison:')
print(f'  Manual     : [{CI_low_b1:.6f}, {CI_high_b1:.6f}]')
print(f'  statsmodels: [{ci_sm.loc["educ", 0]:.6f}, {ci_sm.loc["educ", 1]:.6f}]')
print(f'  Match?     : {np.allclose([CI_low_b1, CI_high_b1], [ci_sm.loc["educ",0], ci_sm.loc["educ",1]])}')

### Step 6.4 — Interpret the CI

In [ ]:
print('INTERPRETATION of the 95% Confidence Interval for beta1:')
print(f'  [{CI_low_b1:.4f}, {CI_high_b1:.4f}]')
print()
print('  We are 95% confident that each additional year of education')
print(f'  raises hourly wages by between ${CI_low_b1:.2f} and ${CI_high_b1:.2f}.')
print()
print('  Does the CI contain 0?', CI_low_b1 <= 0 <= CI_high_b1)
print('  -> NO -> educ is significantly different from 0 at the 5% level.')

---
## Section 7 — Log Transformation

### WHY log(wage) instead of wage?
- Wages are right-skewed; log makes them more symmetric
- Coefficient interpretation becomes a **percentage effect**:
  $\hat{\beta}_1 \approx$ percentage change in wage per extra year of educ

### The log-level model
$$\log(wage_i) = \beta_0 + \beta_1 \cdot educ_i + u_i$$

**Coefficient interpretation:** A one-unit increase in $x$ changes $y$ by approximately $\hat{\beta}_1 \times 100$%.

### Step 7.1 — Create log(wage)

In [ ]:
df['lwage'] = np.log(df['wage'])

print('wage column (first 5):', df['wage'].head().values.round(2))
print('lwage column (first 5):', df['lwage'].head().values.round(4))
print()
print(f'Mean of wage : {df["wage"].mean():.4f}')
print(f'Mean of lwage: {df["lwage"].mean():.4f}')
print(f'Note: exp({df["lwage"].mean():.4f}) = {np.exp(df["lwage"].mean()):.4f} (geometric mean of wage)')

### Step 7.2 — Estimate log-level model manually

In [ ]:
# y is now log(wage)
y_log = df['lwage'].values
x_log = df['educ'].values   # same x as before

y_log_bar = y_log.mean()
x_log_bar = x_log.mean()   # same as x_bar above

x_dev_log = x_log - x_log_bar
y_dev_log = y_log - y_log_bar

beta1_log_manual = np.sum(x_dev_log * y_dev_log) / np.sum(x_dev_log**2)
beta0_log_manual = y_log_bar - beta1_log_manual * x_log_bar

print('Log-level model: log(wage) ~ educ')
print(f'  beta0_hat (manual) = {beta0_log_manual:.6f}')
print(f'  beta1_hat (manual) = {beta1_log_manual:.6f}')

### Step 7.3 — Estimate log-level model with statsmodels

In [ ]:
results_log = smf.ols('lwage ~ educ', data=df).fit()

print('Log-level model: statsmodels output')
print(results_log.summary())

### Step 7.4 — Compare manual vs statsmodels for log-level model

In [ ]:
print('COMPARISON: Log-level model')
print(f'  beta0 manual     : {beta0_log_manual:.6f}')
print(f'  beta0 statsmodels: {results_log.params["Intercept"]:.6f}')
print(f'  Match?           : {np.isclose(beta0_log_manual, results_log.params["Intercept"])}')
print()
print(f'  beta1 manual     : {beta1_log_manual:.6f}')
print(f'  beta1 statsmodels: {results_log.params["educ"]:.6f}')
print(f'  Match?           : {np.isclose(beta1_log_manual, results_log.params["educ"])}')

### Step 7.5 — Interpret the log-level coefficient

In [ ]:
b1_log = results_log.params['educ']

print('INTERPRETATION: log-level model')
print(f'  beta1_hat = {b1_log:.6f}')
print()
print(f'  Approximate interpretation:')
print(f'    Each extra year of education -> +{b1_log*100:.2f}% in wage')
print()
print(f'  Exact interpretation (using exp):')
print(f'    Each extra year of education multiplies wage by exp({b1_log:.4f}) = {np.exp(b1_log):.4f}')
print(f'    -> +{(np.exp(b1_log)-1)*100:.2f}% in wage per extra year of education')
print()
print(f'  Level-level beta1 was ${beta1_hat:.4f}/hour per year')
print(f'  Log-level  beta1 is   {b1_log*100:.2f}% per year')
print(f'  Both capture a positive return to education.')

### Step 7.6 — Compare level-level vs log-level side by side (visual)

In [ ]:
fig, axes = plt.subplots(1, 2, figsize=(13, 5))

x_line = np.linspace(df['educ'].min(), df['educ'].max(), 200)

# Left panel: level-level
axes[0].scatter(df['educ'], df['wage'], alpha=0.3, color='steelblue', s=18)
axes[0].plot(x_line, beta0_hat + beta1_hat*x_line, 'r-', lw=2.5,
    label=f'wage = {beta0_hat:.2f} + {beta1_hat:.3f}*educ')
axes[0].set_title(f'Level-Level Model\nR² = {results.rsquared:.3f}', fontsize=12, fontweight='bold')
axes[0].set_xlabel('Years of Education')
axes[0].set_ylabel('Hourly Wage ($/hour)')
axes[0].legend(fontsize=9)

# Right panel: log-level
axes[1].scatter(df['educ'], df['lwage'], alpha=0.3, color='darkorange', s=18)
axes[1].plot(x_line, beta0_log_manual + beta1_log_manual*x_line, 'r-', lw=2.5,
    label=f'log(wage) = {beta0_log_manual:.2f} + {beta1_log_manual:.4f}*educ')
axes[1].set_title(f'Log-Level Model\nR² = {results_log.rsquared:.3f}', fontsize=12, fontweight='bold')
axes[1].set_xlabel('Years of Education')
axes[1].set_ylabel('log(Hourly Wage)')
axes[1].legend(fontsize=9)

plt.suptitle('Level-Level vs Log-Level: which fits better?', fontsize=13, fontweight='bold')
plt.tight_layout()
plt.savefig(os.path.join(output_dir, 'fig_ch02_02_level_vs_log.png'), dpi=150, bbox_inches='tight')
plt.show()

---
## Section 8 — Visualising the Regression Line and Residual Plot

**Always plot:**
1. The data with the fitted regression line
2. A residual plot (residuals vs fitted values)

The residual plot should look like **random scatter around zero**.
Any pattern = model problem.

### Step 8.1 — Plot the regression line on the scatter

In [ ]:
plt.figure(figsize=(8, 5))
plt.scatter(x, y, alpha=0.35, color='steelblue', s=20, label='Observed data')

x_line = np.linspace(x.min(), x.max(), 200)
plt.plot(x_line, beta0_hat + beta1_hat*x_line, 'r-', lw=2.5,
    label=f'OLS: wage = {beta0_hat:.2f} + {beta1_hat:.3f}*educ')

# Draw a few residual lines to show what OLS minimises
for i in [10, 50, 100, 200, 350]:
    plt.plot([x[i], x[i]], [y[i], y_hat_manual[i]], color='grey', alpha=0.6, lw=1.2)

plt.xlabel('Years of Education', fontsize=12)
plt.ylabel('Hourly Wage ($/hour)', fontsize=12)
plt.title('OLS Regression Line\n(Grey lines = residuals that OLS minimises)', fontsize=12, fontweight='bold')
plt.legend(fontsize=9)
plt.savefig(os.path.join(output_dir, 'fig_ch02_03_ols_line.png'), dpi=150, bbox_inches='tight')
plt.show()

### Step 8.2 — Residual plot (residuals vs fitted values)

In [ ]:
plt.figure(figsize=(8, 5))
plt.scatter(y_hat_manual, resid_manual, alpha=0.35, color='darkorange', s=20)
plt.axhline(0, color='red', lw=2, linestyle='--', label='Zero line')

plt.xlabel('Fitted values (y_hat)', fontsize=12)
plt.ylabel('Residuals (u_hat)', fontsize=12)
plt.title('Residual Plot\n(should be random scatter — no fan, no curve)', fontsize=12, fontweight='bold')
plt.legend()
plt.savefig(os.path.join(output_dir, 'fig_ch02_04_residual_plot.png'), dpi=150, bbox_inches='tight')
plt.show()

print('What to look for:')
print('  Fan shape (wider spread at high fitted values) -> heteroscedasticity (Chapter 8)')
print('  U-shape or curve                              -> nonlinearity (use log or quadratic)')
print('  Random scatter around 0                       -> model is OK')

---
## Section 9 — Monte Carlo: Verifying OLS is Unbiased

### The idea
We just estimated $\hat{\beta}_1 = 0.541$ from ONE sample.
OLS being **unbiased** means: if we could repeat the sampling 10,000 times,
the average of all the $\hat{\beta}_1$ estimates equals the true $\beta_1$.

We simulate this to verify.

### Step 9.1 — Set up the simulation

In [ ]:
np.random.seed(42)

# True data generating process (DGP)
true_beta0 = -0.90
true_beta1 = 0.54    # we pretend this is the 'true' value
true_sigma  = 3.40   # roughly the residual std from our regression

n_obs  = 526    # same as our sample size
n_sims = 10000  # repeat 10,000 times

# Fix the x values (education levels) — same across all simulations
x_fixed = df['educ'].values

print(f'True DGP: wage = {true_beta0} + {true_beta1}*educ + u')
print(f'u ~ N(0, {true_sigma}^2)')
print(f'n_obs = {n_obs}, n_sims = {n_sims:,}')

### Step 9.2 — Run the simulation

In [ ]:
# Store beta1_hat from each simulation
beta1_estimates = np.zeros(n_sims)

for s in range(n_sims):
    # Step 1: Draw new errors (different each simulation)
    u = np.random.normal(0, true_sigma, size=n_obs)

    # Step 2: Generate wage using the true DGP
    wage_sim = true_beta0 + true_beta1 * x_fixed + u

    # Step 3: Estimate OLS (manual formula)
    x_d = x_fixed - x_fixed.mean()
    y_d = wage_sim - wage_sim.mean()
    beta1_estimates[s] = np.sum(x_d * y_d) / np.sum(x_d**2)

print(f'Simulation complete. {n_sims:,} estimates computed.')
print(f'First 10 estimates: {beta1_estimates[:10].round(4)}')

### Step 9.3 — Check if OLS is unbiased

In [ ]:
print('UNBIASEDNESS CHECK:')
print(f'  True beta1                   = {true_beta1}')
print(f'  Mean of {n_sims:,} estimates  = {beta1_estimates.mean():.6f}')
print(f'  Bias = mean - true_beta1     = {beta1_estimates.mean() - true_beta1:.6f}  (should be ~0)')
print(f'  Std of estimates             = {beta1_estimates.std():.6f}')

### Step 9.4 — Visualise the sampling distribution

In [ ]:
plt.figure(figsize=(9, 5))
plt.hist(beta1_estimates, bins=80, density=True, color='steelblue',
    edgecolor='white', alpha=0.85, label='Simulated beta1_hat')
plt.axvline(true_beta1, color='red', lw=2.5, linestyle='--',
    label=f'True beta1 = {true_beta1}')
plt.axvline(beta1_estimates.mean(), color='green', lw=2, linestyle=':',
    label=f'Mean of estimates = {beta1_estimates.mean():.4f}')

# Overlay the theoretical normal distribution
theo_se = true_sigma / np.sqrt(np.sum((x_fixed - x_fixed.mean())**2))
x_range = np.linspace(beta1_estimates.min(), beta1_estimates.max(), 300)
plt.plot(x_range, stats.norm.pdf(x_range, true_beta1, theo_se),
    'orange', lw=2.5, label=f'Theoretical N(beta1, se²)')

plt.xlabel('beta1_hat', fontsize=12)
plt.ylabel('Density', fontsize=12)
plt.title(f'Distribution of OLS Estimates ({n_sims:,} simulations)\n'
    'OLS is UNBIASED: distribution centered on true value', fontsize=12, fontweight='bold')
plt.legend(fontsize=9)
plt.savefig(os.path.join(output_dir, 'fig_ch02_05_MC_unbiasedness.png'), dpi=150, bbox_inches='tight')
plt.show()

### Step 9.5 — Verify with statsmodels on one simulation

In [ ]:
# Take the last simulated sample and run it through statsmodels
# to confirm statsmodels gives the same result as our manual formula
np.random.seed(999)
u_one = np.random.normal(0, true_sigma, size=n_obs)
wage_one = true_beta0 + true_beta1 * x_fixed + u_one
df_one = pd.DataFrame({'wage': wage_one, 'educ': x_fixed})

# Manual
x_d = x_fixed - x_fixed.mean()
y_d = wage_one - wage_one.mean()
b1_manual_one = np.sum(x_d * y_d) / np.sum(x_d**2)

# statsmodels
res_one = smf.ols('wage ~ educ', data=df_one).fit()
b1_sm_one = res_one.params['educ']

print('One simulated sample:')
print(f'  Manual OLS beta1     : {b1_manual_one:.6f}')
print(f'  statsmodels beta1    : {b1_sm_one:.6f}')
print(f'  Match?               : {np.isclose(b1_manual_one, b1_sm_one)}')

---
## Chapter 2 Summary

### Everything we computed, manual vs statsmodels

| Quantity | Manual formula | statsmodels attribute |
|----------|---------------|----------------------|
| beta0_hat | `y_bar - b1*x_bar` | `results.params['Intercept']` |
| beta1_hat | `sum(xd*yd)/sum(xd^2)` | `results.params['educ']` |
| Fitted values | `b0 + b1*x` | `results.fittedvalues` |
| Residuals | `y - y_hat` | `results.resid` |
| SSR | `sum(resid^2)` | `results.ssr` |
| SST | `sum((y-y_bar)^2)` | `results.centered_tss` |
| R² | `SSE/SST` | `results.rsquared` |
| sigma²_hat | `SSR/(n-2)` | `results.mse_resid` |
| se(beta1) | `sqrt(sigma2/SSx)` | `results.bse['educ']` |
| t-stat | `b1/se(b1)` | `results.tvalues['educ']` |
| p-value | `2*t.sf(|t|, n-2)` | `results.pvalues['educ']` |
| 95% CI | `b1 ± t_crit*se` | `results.conf_int()` |

### Key results for `wage ~ educ`
- $\hat{\beta}_1 = 0.541$ → each extra year of education → +\$0.54/hour
- $R^2 = 0.165$ → education explains 16.5% of wage variation
- p-value < 0.001 → highly significant

### Practice Questions
1. Run `wage ~ exper`. Compute beta1 manually. Compare with statsmodels. Interpret.
2. Run `lwage ~ educ`. What is the exact percentage return to education?
3. Compute the 99% CI for beta1 in `wage ~ educ` manually, then verify with statsmodels.
4. In the Monte Carlo, change n_sims to 50,000. Does the mean get closer to true_beta1?
5. What happens to the R² when you use `lwage` instead of `wage`? Can you compare them?